# Etapa 3 — Treinamento da LSTM

Lê `X.npy` e `y.npy`, treina a LSTM e exporta `libras_model.tflite`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'tensorflow', 'scikit-learn', 'matplotlib', '--quiet'], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import itertools

print(f'tensorflow {tf.__version__}')

In [ ]:
SINAIS = ['Oi', 'Tudo_bem', 'Tchau']

X = np.load('X.npy')  # (N, 30, 258)
y = np.load('y.npy')  # (N,)

print(f'X: {X.shape}')
print(f'y: {y.shape}')
print(f'classes: {np.unique(y, return_counts=True)}')

In [ ]:
# one-hot encoding e split 80/20 estratificado
y_cat = keras.utils.to_categorical(y, num_classes=len(SINAIS))

X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y  # garante proporcao igual de cada sinal no treino e validacao
)

print(f'treino:    {X_train.shape}')
print(f'validacao: {X_val.shape}')

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(30, 258)),

    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),

    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),

    layers.LSTM(64),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dense(len(SINAIS), activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [
    # para quando a val_loss nao melhora por 15 epochs e restaura o melhor peso
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),
    # reduz o learning rate pela metade se estagnar por 7 epochs
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
)

In [ ]:
# curvas de loss e acuracia
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='treino')
axes[0].plot(history.history['val_loss'], label='validacao')
axes[0].set_title('loss')
axes[0].set_xlabel('epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='treino')
axes[1].plot(history.history['val_accuracy'], label='validacao')
axes[1].set_title('acuracia')
axes[1].set_xlabel('epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_treino.png', dpi=120)
plt.show()

loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f'\nacuracia na validacao: {acc*100:.1f}%')

In [ ]:
# matriz de confusao
y_pred  = model.predict(X_val, verbose=0)
y_true  = np.argmax(y_val,  axis=1)
y_pred_ = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_true, y_pred_)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ticks = range(len(SINAIS))
ax.set_xticks(list(ticks))
ax.set_yticks(list(ticks))
ax.set_xticklabels([s.replace('_', ' ') for s in SINAIS])
ax.set_yticklabels([s.replace('_', ' ') for s in SINAIS])
ax.set_xlabel('predito')
ax.set_ylabel('real')
ax.set_title('matriz de confusao')

for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, str(cm[i, j]), ha='center', va='center',
            color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=120)
plt.show()

print(classification_report(y_true, y_pred_,
                             target_names=[s.replace('_',' ') for s in SINAIS]))

In [ ]:
# exporta para TFLite com quantizacao float16
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open('libras_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'libras_model.tflite salvo  {len(tflite_model)/1024:.0f} KB')

In [ ]:
# teste rapido do modelo TFLite com uma amostra da validacao
interp = tf.lite.Interpreter(model_path='libras_model.tflite')
interp.allocate_tensors()

inp = interp.get_input_details()
out = interp.get_output_details()

amostra = X_val[:1].astype(np.float32)  # (1, 30, 258)
interp.set_tensor(inp[0]['index'], amostra)
interp.invoke()
saida = interp.get_tensor(out[0]['index'])[0]

idx       = np.argmax(saida)
confianca = saida[idx]
real      = SINAIS[np.argmax(y_val[0])]

print(f'real:     {real}')
print(f'predito:  {SINAIS[idx]}  ({confianca*100:.1f}%)')
print(f'saidas:   { {SINAIS[i]: f"{saida[i]:.3f}" for i in range(len(SINAIS))} }')